# Preprocesado del dataset de ejercicios

Este notebook limpia el archivo `context/megaGymDataset.csv` y genera únicamente los archivos realmente necesarios para el generador de rutinas con IA.

Archivos generados:

- `processed_context/exercises_clean.csv`
- `processed_context/exercises_catalog.json`
- `processed_context/catalog_summary.json`

No se generan archivos separados por músculo, nivel o material, porque esa información ya está dentro del catálogo principal y se puede filtrar desde Python cuando haga falta.


In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np

# Rutas del proyecto
INPUT_PATH = Path("context") / "megaGymDataset.csv"
OUTPUT_DIR = Path("processed_context")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ruta de entrada:", INPUT_PATH)
print("Carpeta de salida:", OUTPUT_DIR.resolve())


Ruta de entrada: context\megaGymDataset.csv
Carpeta de salida: C:\Users\carlo\Desktop\TFGampliado\processed_context


In [2]:
# Cargar dataset original
df_raw = pd.read_csv(INPUT_PATH)

print("Dimensiones originales:", df_raw.shape)
display(df_raw.head())


Dimensiones originales: (2918, 9)


,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


In [3]:
# Revisar columnas y nulos principales
print("Columnas originales:")
print(df_raw.columns.tolist())

print("\nValores nulos por columna:")
display(df_raw.isna().sum().sort_values(ascending=False))


Columnas originales:
['Unnamed: 0', 'Title', 'Desc', 'Type', 'BodyPart', 'Equipment', 'Level', 'Rating', 'RatingDesc']

Valores nulos por columna:


RatingDesc    2056
Rating        1887
Desc          1550
Equipment       32
Unnamed: 0       0
BodyPart         0
Type             0
Title            0
Level            0
dtype: int64

## 1. Limpieza básica

Se eliminan columnas que no aportan valor directo al generador:

- `Unnamed: 0`: índice antiguo.
- `RatingDesc`: aporta poca información útil en este dataset.

También se renombran las columnas para trabajar con nombres más claros y consistentes.


In [4]:
df = df_raw.copy()

# Eliminar columnas innecesarias si existen
columns_to_drop = ["Unnamed: 0", "RatingDesc"]
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors="ignore")

# Normalizar nombres de columnas
df = df.rename(columns={
    "Title": "name",
    "Desc": "description",
    "Type": "type",
    "BodyPart": "body_part",
    "Equipment": "equipment",
    "Level": "level",
    "Rating": "rating"
})

# Asegurar que existen todas las columnas esperadas
expected_columns = ["name", "description", "type", "body_part", "equipment", "level", "rating"]
missing_columns = [col for col in expected_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Faltan columnas esperadas en el dataset: {missing_columns}")

df = df[expected_columns].copy()

display(df.head())
print(df.columns.tolist())


,name,description,type,body_part,equipment,level,rating
0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN
3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN


['name', 'description', 'type', 'body_part', 'equipment', 'level', 'rating']


## 2. Normalización de texto y valores numéricos

Se limpian espacios, saltos de línea y valores vacíos.  
El campo `rating` se convierte a número y los valores vacíos pasan a `0.0`.


In [5]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    return value

text_columns = ["name", "description", "type", "body_part", "equipment", "level"]

for col in text_columns:
    df[col] = df[col].apply(clean_text)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce").fillna(0.0)

display(df.head())


,name,description,type,body_part,equipment,level,rating
0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,0.0
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,0.0
3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,0.0
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,0.0


## 3. Eliminación de ejercicios inválidos y duplicados

Se eliminan:

- filas sin nombre de ejercicio;
- duplicados por combinación de nombre, tipo, grupo muscular, material y nivel.

Esto evita que el catálogo contenga entradas repetidas que puedan confundir a la IA.


In [6]:
before = len(df)

# Eliminar filas sin nombre
df = df[df["name"].str.len() > 0].copy()

# Eliminar duplicados por campos clave
dedup_cols = ["name", "type", "body_part", "equipment", "level"]
df = df.drop_duplicates(subset=dedup_cols, keep="first").copy()

after = len(df)

print(f"Filas antes: {before}")
print(f"Filas después: {after}")
print(f"Filas eliminadas: {before - after}")


Filas antes: 2918
Filas después: 2909
Filas eliminadas: 9


## 4. Creación de identificadores únicos

Cada ejercicio recibe un `exercise_id` estable.  
Este ID será importante para validar que la IA solo usa ejercicios existentes en el catálogo.


In [7]:
def slugify(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "exercise"

df = df.reset_index(drop=True)

base_ids = df["name"].apply(slugify)
seen = {}
exercise_ids = []

for base_id in base_ids:
    count = seen.get(base_id, 0) + 1
    seen[base_id] = count

    if count == 1:
        exercise_ids.append(base_id)
    else:
        exercise_ids.append(f"{base_id}_{count}")

df.insert(0, "exercise_id", exercise_ids)

display(df.head())
print("IDs únicos:", df["exercise_id"].is_unique)


,exercise_id,name,description,type,body_part,equipment,level,rating
0,partner_plank_band_row,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0
1,banded_crunch_isometric_hold,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,0.0
2,fyr_banded_plank_jack,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,0.0
3,banded_crunch,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,0.0
4,crunch,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,0.0


IDs únicos: True


## 5. Revisión rápida del catálogo limpio

In [8]:
print("Total de ejercicios limpios:", len(df))

print("\nTipos de ejercicio:")
display(df["type"].value_counts())

print("\nGrupos musculares:")
display(df["body_part"].value_counts())

print("\nMaterial:")
display(df["equipment"].value_counts())

print("\nNivel:")
display(df["level"].value_counts())


Total de ejercicios limpios: 2909

Tipos de ejercicio:


type
Strength                 2536
Stretching                147
Plyometrics                97
Powerlifting               37
Cardio                     35
Olympic Weightlifting      35
Strongman                  22
Name: count, dtype: int64


Grupos musculares:


body_part
Abdominals     660
Quadriceps     645
Shoulders      338
Chest          260
Biceps         168
Triceps        151
Lats           124
Hamstrings     121
Middle Back    116
Lower Back      97
Glutes          81
Calves          47
Forearms        31
Traps           24
Abductors       21
Adductors       17
Neck             8
Name: count, dtype: int64


Material:


equipment
Body Only        1078
Dumbbell          513
Barbell           281
Other             254
Cable             223
Machine           175
Kettlebells       149
Bands              98
Medicine Ball      38
Exercise Ball      35
                   32
E-Z Curl Bar       22
Foam Roll          11
Name: count, dtype: int64


Nivel:


level
Intermediate    2437
Beginner         459
Expert            13
Name: count, dtype: int64

## 6. Exportación final

Se generan solo tres archivos:

1. `exercises_clean.csv`: útil para revisar el catálogo manualmente.
2. `exercises_catalog.json`: catálogo principal que usará la aplicación/IA.
3. `catalog_summary.json`: resumen estadístico para inspección y documentación.

No se generan archivos agrupados porque son redundantes.


In [9]:
# Rutas de salida
clean_csv_path = OUTPUT_DIR / "exercises_clean.csv"
catalog_json_path = OUTPUT_DIR / "exercises_catalog.json"
summary_json_path = OUTPUT_DIR / "catalog_summary.json"

# Exportar CSV limpio
df.to_csv(clean_csv_path, index=False, encoding="utf-8")

# Exportar catálogo principal en JSON
records = df.to_dict(orient="records")

with open(catalog_json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

# Exportar resumen
summary = {
    "total_exercises": int(len(df)),
    "columns": df.columns.tolist(),
    "types": df["type"].value_counts().to_dict(),
    "body_parts": df["body_part"].value_counts().to_dict(),
    "equipment": df["equipment"].value_counts().to_dict(),
    "levels": df["level"].value_counts().to_dict(),
    "notes": [
        "Dataset preparado como catálogo cerrado de ejercicios.",
        "La IA debe seleccionar ejercicios existentes por exercise_id o name.",
        "El campo description puede estar vacío en algunos ejercicios.",
        "El campo rating se ha normalizado a 0.0 cuando estaba vacío.",
        "No se generan archivos agrupados porque el filtrado se puede hacer desde el catálogo principal."
    ]
}

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Archivos generados:")
print("-", clean_csv_path)
print("-", catalog_json_path)
print("-", summary_json_path)


Archivos generados:
- processed_context\exercises_clean.csv
- processed_context\exercises_catalog.json
- processed_context\catalog_summary.json


## 7. Comprobación de archivos generados

In [10]:
generated_files = sorted(OUTPUT_DIR.glob("*"))

print("Contenido actual de processed_context:")
for file in generated_files:
    print("-", file.name)

print("\nArchivos necesarios para el proyecto:")
print("- exercises_clean.csv")
print("- exercises_catalog.json")
print("- catalog_summary.json")


Contenido actual de processed_context:
- catalog_summary.json
- exercises_catalog.json
- exercises_clean.csv

Archivos necesarios para el proyecto:
- exercises_clean.csv
- exercises_catalog.json
- catalog_summary.json


## Uso recomendado

Para el generador de rutinas, utiliza principalmente:

```text
processed_context/exercises_catalog.json
```

Ese archivo debe actuar como catálogo cerrado.  
La IA puede proponer rutinas, pero después tu código debe validar que cada `exercise_id` existe en este catálogo.

El CSV queda solo como apoyo para revisar datos manualmente.
